# Session 5: reading trajectories

Exploration only. Run `make agent-eval` first so `artifacts/traces` and `artifacts/agent_eval.json` exist.

## Every run, at a glance

In [ ]:
import json
from pathlib import Path
from nw.agent.trace import Trajectory, replay
traces = [Trajectory.load(p) for p in sorted(Path("../artifacts/traces").glob("*.json"))]
print(len(traces), "runs")
for t in traces[:20]:
    print(f"{t.run_id}  {t.agent:12s} {t.terminated.value:9s} steps={t.n_steps:2d} cost={t.cost_usd:.4f}  tools={t.tools_called}")

## The failures, and the step where each went wrong

In [ ]:
ev = json.load(open("../artifacts/agent_eval.json"))
failed = [s for s in ev["scores"] if not s["success"]]
print(len(failed), "failed of", ev["aggregate"]["n"])
for s in failed:
    print("\n==", s["id"], s["kind"], s["failures"])
    t = Trajectory.load(Path("../artifacts/traces") / f"{s['trace']}.json")
    print(replay(t)[:1500])

## Cost per resolution by kind

In [ ]:
import collections, statistics
by_kind = collections.defaultdict(list)
for s in ev["scores"]:
    by_kind[s["kind"]].append(s["cost_usd"])
for k, v in sorted(by_kind.items(), key=lambda kv: -statistics.mean(kv[1])):
    print(f"{k:16s} n={len(v)} mean {statistics.mean(v):.4f} USD max {max(v):.4f}")
print("overall per resolution:", round(ev["aggregate"]["cost_usd_per_resolution"], 4), "USD")

## Questions to bring back

- For each failure: tool design, prompt, or the case itself?
- What would the cost per resolution be at Northwind's 8,000 tickets a quarter, and who signs off on that?